# Fill in missing ratings and choose the rank

A ratings matrix has one row per user and one column per item, and many entries are
missing. If tastes come from a few hidden factors, the full matrix is close to low rank, and
that structure is enough to guess the gaps: look for a matrix of low rank that agrees with
every rating you have. How low is the question. Nobody tells you how many hidden factors
there are, so you will pick the rank from the data, by hiding some ratings you know and
checking how well each rank recovers them.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
# 40 users rate 25 items. The scores come from a few hidden tastes (you are not told how
# many). About 40 percent of the entries were never observed (np.nan).
users = rng.normal(size=(40, 3))
items = rng.normal(size=(3, 25))
R_true = users @ items
observed = rng.random(R_true.shape) < 0.6
R_obs = np.where(observed, R_true, np.nan)
print("observed entries:", int(observed.sum()), "of", R_true.size)

## Part 1 (coding task): How many singular values hold the energy

The energy of a matrix is the sum of its squared singular values, $\lVert A \rVert_F^2$.
Write `energy_rank(s, frac)`: given the singular values `s` in decreasing order, return the
smallest $k$ such that the first $k$ hold at least the fraction `frac` of the energy,
$\sigma_1^2 + \dots + \sigma_k^2 \ge \text{frac} \cdot \sum_i \sigma_i^2$.

In [ ]:
def energy_rank(s, frac):
    s = np.asarray(s, dtype=float)
    # TODO: smallest k whose first k squared singular values reach frac of the total.
    raise NotImplementedError

In [ ]:
assert energy_rank([5.0, 3.0, 1.0, 1.0], 0.9) == 2, f"25 + 9 = 34 of 36 is over 90 percent and 25 alone is not, so k = 2, got {energy_rank([5.0, 3.0, 1.0, 1.0], 0.9)}: square the singular values before adding, and count k from 1"
assert energy_rank([5.0, 3.0, 1.0, 1.0], 0.95) == 3, "35 of 36 is the first total over 95 percent, so k = 3"
assert energy_rank([1.0, 1.0], 0.5) == 1, "exactly half the energy counts as reaching 0.5: use >=, and count k from 1"
assert energy_rank([4.0, 0.0, 0.0], 1.0) == 1, "all the energy is in the first singular value"
filled = np.where(observed, R_obs, 0.0)
s_filled = np.linalg.svd(filled, compute_uv=False)
print("singular values with the gaps set to 0:", np.round(s_filled[:6], 2))
print("rank for 90 percent of that energy:", energy_rank(s_filled, 0.9))
print("Part 1 passed")

## Part 2 (coding task): The best rank-k approximation

Write `truncate(A, k)`, the best rank-$k$ approximation of `A`: keep the $k$ terms of its
SVD that have the largest singular values. Call `np.linalg.svd` with `full_matrices=False`
so the factors have only as many columns as needed.

In [ ]:
def truncate(A, k):
    A = np.asarray(A, dtype=float)
    # TODO: keep the k largest terms of the SVD of A.
    raise NotImplementedError

In [ ]:
T = np.random.default_rng(11).normal(size=(8, 5))
sv = np.linalg.svd(T, compute_uv=False)
T2 = np.asarray(truncate(T, 2), dtype=float)
assert T2.shape == T.shape, f"the approximation should have the shape of the input, {T.shape}, got {T2.shape}"
assert np.linalg.matrix_rank(T2) == 2, f"truncate(T, 2) should have rank 2, got {np.linalg.matrix_rank(T2)}"
err = np.linalg.norm(T - T2)
assert np.isclose(err, np.sqrt((sv[2:] ** 2).sum())), f"the error should be the root of the dropped squared singular values, {np.sqrt((sv[2:] ** 2).sum()):.4f}, got {err:.4f}: keep the largest terms, each scaled by its singular value"
assert np.allclose(truncate(T, 5), T), "keeping every term should give the matrix back"
print("Part 2 passed")

## Part 3 (coding task): Fill the gaps at a given rank

Write `complete(R_obs, k, steps=300)`. `R_obs` has `np.nan` where a rating is missing.
The completed matrix should meet two requirements at once: rank at most $k$, and agreement
with every known rating. `truncate` enforces the first and breaks the second, and writing the
known ratings back in does the opposite. Put 0 in every gap to begin, alternate the two for
`steps` rounds, and return the full matrix that holds the known ratings exactly.

In [ ]:
def complete(R_obs, k, steps=300):
    R_obs = np.asarray(R_obs, dtype=float)
    # TODO: alternate between rank at most k and agreeing with the known ratings.
    raise NotImplementedError

In [ ]:
X = np.asarray(complete(R_obs, 3), dtype=float)
assert X.shape == R_obs.shape, f"expected shape {R_obs.shape}, got {X.shape}"
assert not np.isnan(X).any(), "the completed matrix still has nan entries"
assert np.allclose(X[observed], R_obs[observed]), "the known ratings must come back exactly as given: write them back after every truncation"
rmse = np.sqrt(np.mean((X - R_true)[~observed] ** 2))
assert rmse < 1e-3, f"at k = 3 the error on the missing entries is {rmse:.4f}: keep the known entries fixed by position (from R_obs), and truncate again every round"
u3, v3 = np.random.default_rng(21).normal(size=(10, 2)), np.random.default_rng(22).normal(size=(2, 8))
R3_true = u3 @ v3
m3 = np.random.default_rng(23).random((10, 8)) < 0.7
R3 = np.where(m3, R3_true, np.nan)
X3 = np.asarray(complete(R3, 2), dtype=float)
assert np.allclose(X3[m3], R3[m3]), "the known ratings must come back exactly on any low-rank matrix, not just the one in the setup cell"
assert np.sqrt(np.mean((X3 - R3_true)[~m3] ** 2)) < 1e-3, "complete must recover a rank-2 matrix it has never seen: the ground truth of the setup cell is not an acceptable answer"
print(f"k = 3: error on the missing entries {rmse:.2e}")
print("Part 3 passed")

## Part 4 (coding task): Choose the rank with hidden ratings

Write `choose_rank(R_obs, ranks, holdout=0.2, seed=0)`. Hide a random fraction `holdout` of
the known ratings (use `np.random.default_rng(seed)`), complete what is left with each rank
in `ranks`, and score each rank by the root mean squared error on the hidden ratings. Return
the rank with the smallest error.

In [ ]:
def choose_rank(R_obs, ranks, holdout=0.2, seed=0):
    R_obs = np.asarray(R_obs, dtype=float)
    # TODO: hide some known ratings, complete at each rank, score on the hidden ones.
    raise NotImplementedError

In [ ]:
k_best = choose_rank(R_obs, range(1, 7))
assert k_best == 3, f"choose_rank picked k = {k_best}: score each rank on ratings the completion did not see. The known ratings it was given come back exactly at every rank."
users2, items2 = np.random.default_rng(3).normal(size=(30, 2)), np.random.default_rng(4).normal(size=(2, 20))
mask2 = np.random.default_rng(5).random((30, 20)) < 0.7
R2 = np.where(mask2, users2 @ items2, np.nan)
k2 = choose_rank(R2, range(1, 6), seed=1)
assert k2 == 2, f"on a second matrix built from 2 tastes, choose_rank picked {k2}"
for k in (2, 3, 4):
    Xk = complete(R_obs, k)
    print(f"k = {k}: error on the missing entries {np.sqrt(np.mean((Xk - R_true)[~observed] ** 2)):.4f}")
print("Part 4 passed")

## Solution

### Part 1: How many singular values hold the energy

In [ ]:
def energy_rank(s, frac):
    s = np.asarray(s, dtype=float)
    energy = np.cumsum(s ** 2)                 # energy[k - 1]: the first k singular values
    return int(np.argmax(energy >= frac * energy[-1]) + 1)

**Why it works.** 
`np.cumsum(s ** 2)` lists the energy of the first 1, 2, 3, ... singular values, and the
first index where it reaches the target, plus one, is $k$. Adding the singular values
themselves is the common slip: the Frobenius norm squares them, so a few large ones hold
more of the energy than their plain sum suggests. The printout shows why energy alone cannot
pick the rank for incomplete data: zeros in the gaps are not low rank, so the singular values
of the filled matrix trail off slowly and 90 percent needs far more than the true number of
factors. Part 4 picks the rank another way.

### Part 2: The best rank-k approximation

In [ ]:
def truncate(A, k):
    left, sig, right_t = np.linalg.svd(np.asarray(A, dtype=float), full_matrices=False)
    return (left[:, :k] * sig[:k]) @ right_t[:k]       # sig is in decreasing order

**Why it works.** 
`svd` returns the singular values from largest to smallest, so the first $k$ columns of
`left`, entries of `sig` and rows of `right_t` are the $k$ largest terms.
`left[:, :k] * sig[:k]` scales column $i$ by $\sigma_i$, and multiplying by `right_t[:k]`
adds up the $k$ outer products, one per kept singular value. By
Eckart-Young no rank-$k$ matrix is closer in the Frobenius norm, and the error is
$\sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \dots}$.

### Part 3: Fill the gaps at a given rank

In [ ]:
def complete(R_obs, k, steps=300):
    R_obs = np.asarray(R_obs, dtype=float)
    known = ~np.isnan(R_obs)
    X = np.where(known, R_obs, 0.0)
    for _ in range(steps):
        X = np.where(known, R_obs, truncate(X, k))     # rank-k values in the gaps only
    return X

**Why it works.** 
Each round replaces the matrix by the closest rank-$k$ matrix, then by the closest matrix
that agrees with the known ratings. Neither move can increase the distance between the
current matrix and the nearest rank-$k$ matrix, so the rounds settle. With the right rank they
settle on the rank-3 matrix that agrees with every rating: the 608 known ratings are far more
than the $3(40 + 25 - 3) = 186$ numbers that pin down a rank-3 matrix of this shape. The
positions of the gaps have to come from `R_obs`: after the first round the working matrix has
no `nan` left, so looking for gaps in it finds none and nothing is ever updated.

### Part 4: Choose the rank with hidden ratings

In [ ]:
def choose_rank(R_obs, ranks, holdout=0.2, seed=0):
    R_obs = np.asarray(R_obs, dtype=float)
    known = ~np.isnan(R_obs)
    hide = known & (np.random.default_rng(seed).random(R_obs.shape) < holdout)
    train = np.where(hide, np.nan, R_obs)                # the hidden ratings become gaps
    errors = {}
    for k in ranks:
        X = complete(train, k)
        errors[k] = np.sqrt(np.mean((X - R_obs)[hide] ** 2))
    return min(errors, key=errors.get)

**Why it works.** 
A rank that is too small cannot fit the ratings, and a rank that is too large has room to fit
them with wrong values in the gaps: the printout shows both, against an error near 0 at the
true rank. The known ratings cannot tell these apart, because every completion returns them
exactly, so scoring on them gives an error of 0 at every rank. Hiding some of them turns them
into a validation set, the same idea as choosing any hyperparameter on held-out data. The
hidden ratings must be taken out of the input to `complete`, or the method sees the answers.